# 分箱修复后复核（2026-10-27）

保留修复前审计 Notebook 和 JSON，此文件重新执行同一真实数据基线、留出比较和反例。默认已关闭额外 Lift 调整；显式开启和关闭分别记录。不可行约束的异常是预期行为，不作为算法成功结果。

使用仓库现有验收解释器，不安装或升级环境。所有对照来自真实运行，原始客户行不写入结果文件。

In [1]:
import ast, hashlib, inspect, json, platform, sys, time, warnings
from pathlib import Path
from importlib import metadata
from unittest.mock import patch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics import compute_bin_stats
import hscredit.core.binning as hbin
import toad
from optbinning import OptimalBinning as OptBinning

ROOT = Path.cwd()
if not (ROOT / 'hscredit/core/binning').is_dir():
    ROOT = next(p for p in ROOT.parents if (p / 'hscredit/core/binning').is_dir())
OUT = ROOT / 'docs/validation'
OUT.mkdir(exist_ok=True)
DATA = pd.read_excel(ROOT / 'examples/hscredit_yyp.xlsx')
FEATURES = ['衡枢鉴真分老客版', '近六个月非银多头机构数', '青云24']
COMMON = dict(max_n_bins=5, min_n_bins=2, min_bin_size=0.05, monotonic=False, random_state=42, n_jobs=1)
VERSIONS = {name: metadata.version(name) for name in ['hscredit','numpy','pandas','scikit-learn','ortools','toad','optbinning']}
EVIDENCE = {'日期':'2026-10-27','时区':'Asia/Shanghai','解释器':sys.executable,'Python':platform.python_version(),'依赖版本':VERSIONS,
    '数据行数':len(DATA),'数据列数':len(DATA.columns),'目标分布':DATA.FPD.value_counts().to_dict(),
    '源码SHA256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((ROOT/'hscredit/core/binning').glob('*.py'))}}
print(json.dumps({k:v for k,v in EVIDENCE.items() if k!='源码SHA256'},ensure_ascii=False,indent=2))
assert len(DATA)==970 and set(DATA.FPD.unique())=={0,1}


{
  "日期": "2026-10-27",
  "时区": "Asia/Shanghai",
  "解释器": "E:\\workspace\\hscredit\\.audit_tmp\\defect-repairs\\runtime\\Scripts\\python.exe",
  "Python": "3.11.5",
  "依赖版本": {
    "hscredit": "0.1.3",
    "numpy": "2.4.6",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "ortools": "9.7.2996",
    "toad": "0.1.4",
    "optbinning": "0.20.1"
  },
  "数据行数": 970,
  "数据列数": 18,
  "目标分布": {
    "0": 834,
    "1": 136
  }
}


## 统一统计口径与独立复核

所有库的最终箱号都交给 `compute_bin_stats(round_digits=False)`，避免把各库 WOE 平滑口径差别误判为分箱质量差别；逐箱样本数、坏数由原始行单独聚合核对，无纯好/纯坏箱时另以原始计数公式核验 IV。

In [2]:
def aggregate(bins, y):
    bins, y = np.asarray(bins, dtype=int), np.asarray(y, dtype=int)
    stats = compute_bin_stats(bins, y, round_digits=False)
    raw = pd.DataFrame({'分箱':bins,'目标':y}).groupby('分箱',sort=True)['目标'].agg(['size','sum'])
    got = stats.set_index('分箱').loc[raw.index]
    np.testing.assert_array_equal(got['样本总数'], raw['size'])
    np.testing.assert_array_equal(got['坏样本数'], raw['sum'])
    assert raw['size'].sum()==len(y)
    good = raw['size'].to_numpy()-raw['sum'].to_numpy()
    bad = raw['sum'].to_numpy()
    iv = float(stats['分档IV值'].sum())
    if (good>0).all() and (bad>0).all():
        p, q = bad/bad.sum(),good/good.sum()
        independent_iv = float(np.sum((p-q)*np.log(p/q)))
        np.testing.assert_allclose(iv,independent_iv,rtol=1e-8,atol=1e-10)
    regular = raw.loc[raw.index>=0]
    return {'实际箱数':len(regular),'最小箱占比':float(regular['size'].min()/len(y)),
        '最大箱占比':float(regular['size'].max()/len(y)), 'IV':iv,
        'KS':float(stats['分档KS值'].max()),'箱样本数':regular['size'].astype(int).tolist(),
        '箱坏样本数':regular['sum'].astype(int).tolist(),'箱坏率':(regular['sum']/regular['size']).tolist()}

def clean_json(value):
    if isinstance(value,dict): return {str(k):clean_json(v) for k,v in value.items()}
    if isinstance(value,(list,tuple,np.ndarray)): return [clean_json(v) for v in value]
    if isinstance(value,np.generic): return clean_json(value.item())
    if isinstance(value,float) and not np.isfinite(value): return None
    return value

def save_evidence():
    (OUT/'binning-method-repair-2026-10-27.json').write_text(json.dumps(clean_json(EVIDENCE),ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')

ROWS = []
def record_fit(name, feature, make, family='hscredit'):
    x = DATA[feature].to_numpy()
    y = DATA.FPD.to_numpy()
    start = time.perf_counter()
    row = {'方法':name,'字段':feature,'库':family}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        try:
            binner = make()
            if family=='hscredit':
                binner.fit(DATA[[feature]],DATA.FPD)
                bins=binner.transform(DATA[[feature]])[feature].to_numpy()
                row['切点']=binner.splits_[feature].tolist()
            elif family=='toad':
                bins,splits=binner(x,y)
                row['切点']=splits.tolist()
            else:
                binner.fit(x,y)
                bins=binner.transform(x,metric='indices')
                row['切点']=binner.splits.tolist()
                row['求解状态']=binner.status
            row.update(aggregate(bins,y))
        except Exception as error:
            row['错误']=type(error).__name__+': '+str(error)
        row['警告']=list(dict.fromkeys(str(w.message) for w in caught))
    row['耗时秒']=time.perf_counter()-start
    ROWS.append(row)
    EVIDENCE['真实数据完整样本比较']=ROWS
    save_evidence()


In [3]:
for feature in FEATURES:
    for method in OptimalBinning.VALID_METHODS:
        for refine in [True,False]:
            record_fit(method+('/启用调整' if refine else '/关闭调整'),feature,
                lambda m=method,r=refine:OptimalBinning(method=m,lift_refine=r,**COMMON))
    for method in ['dt','chi','quantile','step','kmeans']:
        kwargs={'n_bins':5}
        if method=='dt': kwargs.update(min_samples=0.05,random_state=42)
        if method=='kmeans': kwargs.update(random_state=42)
        record_fit(method,feature,lambda m=method,k=kwargs:lambda x,y:toad.merge(x,y,method=m,return_splits=True,**k),family='toad')
    for trend in [None,'auto']:
        record_fit('最优IV/'+str(trend),feature,
            lambda t=trend:OptBinning(max_n_bins=5,min_n_bins=2,min_bin_size=0.05,
                max_n_prebins=20,min_prebin_size=0.05,monotonic_trend=t,time_limit=30),family='optbinning')
    print('完成字段：',feature)
    print(pd.DataFrame([r for r in ROWS if r['字段']==feature]).reindex(columns=['库','方法','实际箱数','最小箱占比','IV','KS','错误']).to_string(index=False))


  File "C:\Users\Administrator\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Administrator\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


完成字段： 衡枢鉴真分老客版
         库                   方法  实际箱数  最小箱占比     IV     KS  错误
  hscredit         uniform/启用调整     4 0.0557 0.2254 0.1748 NaN
  hscredit         uniform/关闭调整     4 0.0557 0.2254 0.1748 NaN
  hscredit        quantile/启用调整     5 0.2000 0.1859 0.1693 NaN
  hscredit        quantile/关闭调整     5 0.2000 0.1859 0.1693 NaN
  hscredit            tree/启用调整     5 0.0505 0.2793 0.1813 NaN
  hscredit            tree/关闭调整     5 0.0505 0.3368 0.1789 NaN
  hscredit             chi/启用调整     5 0.0505 0.2102 0.1431 NaN
  hscredit             chi/关闭调整     4 0.0907 0.1249 0.1431 NaN
  hscredit         best_ks/启用调整     5 0.0505 0.2725 0.1813 NaN
  hscredit         best_ks/关闭调整     5 0.0505 0.3181 0.1813 NaN
  hscredit         best_iv/启用调整     5 0.0505 0.2879 0.1739 NaN
  hscredit         best_iv/关闭调整     5 0.0598 0.3177 0.1739 NaN
  hscredit            mdlp/启用调整     5 0.0505 0.2579 0.1813 NaN
  hscredit            mdlp/关闭调整     2 0.1320 0.1998 0.1715 NaN
  hscredit        or_tools/启用调整     5 0.

完成字段： 近六个月非银多头机构数
         库                   方法  实际箱数  最小箱占比     IV     KS  错误
  hscredit         uniform/启用调整     4 0.0722 0.0301 0.0595 NaN
  hscredit         uniform/关闭调整     4 0.0722 0.0301 0.0595 NaN
  hscredit        quantile/启用调整     5 0.1680 0.0649 0.1036 NaN
  hscredit        quantile/关闭调整     5 0.1680 0.0649 0.1036 NaN
  hscredit            tree/启用调整     3 0.0619 0.0754 0.1093 NaN
  hscredit            tree/关闭调整     5 0.0619 0.1009 0.1093 NaN
  hscredit             chi/启用调整     3 0.0814 0.0721 0.1036 NaN
  hscredit             chi/关闭调整     2 0.1309 0.0680 0.0957 NaN
  hscredit         best_ks/启用调整     3 0.0794 0.0692 0.1093 NaN
  hscredit         best_ks/关闭调整     5 0.0577 0.1178 0.1093 NaN
  hscredit         best_iv/启用调整     3 0.0794 0.0692 0.1093 NaN
  hscredit         best_iv/关闭调整     5 0.0577 0.1178 0.1093 NaN
  hscredit            mdlp/启用调整     3 0.0794 0.0692 0.1093 NaN
  hscredit            mdlp/关闭调整     2 0.1928 0.0680 0.1093 NaN
  hscredit        or_tools/启用调整     5

完成字段： 青云24
         库                   方法  实际箱数  最小箱占比     IV     KS  错误
  hscredit         uniform/启用调整     3 0.1876 0.0382 0.0643 NaN
  hscredit         uniform/关闭调整     3 0.1876 0.0382 0.0643 NaN
  hscredit        quantile/启用调整     5 0.1948 0.0546 0.0930 NaN
  hscredit        quantile/关闭调整     5 0.1948 0.0546 0.0930 NaN
  hscredit            tree/启用调整     4 0.2041 0.0866 0.1375 NaN
  hscredit            tree/关闭调整     5 0.0794 0.2586 0.1375 NaN
  hscredit             chi/启用调整     4 0.1505 0.0475 0.0715 NaN
  hscredit             chi/关闭调整     3 0.2825 0.0167 0.0622 NaN
  hscredit         best_ks/启用调整     4 0.1959 0.0808 0.1363 NaN
  hscredit         best_ks/关闭调整     5 0.0784 0.2147 0.1363 NaN
  hscredit         best_iv/启用调整     4 0.1959 0.0808 0.1363 NaN
  hscredit         best_iv/关闭调整     5 0.0784 0.2147 0.1363 NaN
  hscredit            mdlp/启用调整     2 0.4711 0.0780 0.1375 NaN
  hscredit            mdlp/关闭调整     2 0.4711 0.0780 0.1375 NaN
  hscredit        or_tools/启用调整     5 0.0536

## 结构和约束反例复核

排序后的同一总体、低基数和精度样本应返回一致、有效的规则；不可行方向和人数约束必须明确失败。保留聚合结果与异常，末尾统一断言。

In [4]:
CASES=[]
def sample_by_counts(bad_counts,counts=None):
    counts = counts if counts is not None else [20]*len(bad_counts)
    x = np.repeat(np.arange(len(bad_counts),dtype=float),counts)
    y = np.concatenate([np.r_[np.ones(b,dtype=int),np.zeros(n-b,dtype=int)] for b,n in zip(bad_counts,counts)])
    return pd.DataFrame({'变量':x}),pd.Series(y)

def fit_case(case, binner, X, y):
    row={'实验':case,'分箱器':type(binner).__name__}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        try:
            binner.fit(X,y)
            name=X.columns[0]
            row['切点']=clean_json(binner.splits_[name])
            row['声明箱数']=binner.n_bins_[name]
            row.update(aggregate(binner.transform(X)[name],y))
        except Exception as error:
            row['错误']=type(error).__name__+': '+str(error)
        row['警告']=list(dict.fromkeys(str(w.message) for w in caught))
    CASES.append(row)
    return row

# 1. 同一组 (x,y) 仅改变排列顺序。
X,y=sample_by_counts([2,3,4,5,6,8,10,12,13,15,16,18])
for method in ['cp_sat','or_tools']:
    for ordering in ['原始','乱序','倒序']:
        order=np.arange(len(y))
        if ordering=='乱序': order=np.random.default_rng(42).permutation(order)
        if ordering=='倒序': order=order[::-1]
        params=dict(method=method,lift_refine=False,max_n_bins=4,min_n_bins=2,min_bin_size=0.1,monotonic=False,n_jobs=1)
        if method=='or_tools': params['use_cp_sat']=True
        fit_case('行顺序/'+method+'/'+ordering,OptimalBinning(**params),X.iloc[order].reset_index(drop=True),y.iloc[order].reset_index(drop=True))

# 2. 三种数值取值足以产生三个合规箱；却退化到一箱或漏掉值间边界。
X,y=sample_by_counts([10,50,90],[100,100,100])
for method in ['quantile','chi','kmeans','best_iv','best_ks','cp_sat','or_tools']:
    fit_case('低基数/'+method,OptimalBinning(method=method,lift_refine=False,max_n_bins=5,min_n_bins=2,min_bin_size=0.1,n_jobs=1,random_state=42),X,y)
fit_case('低基数/最大箱40%',OptimalBinning(method='cp_sat',lift_refine=False,max_n_bins=5,min_n_bins=2,min_bin_size=0.1,max_bin_size=0.4,monotonic=False,n_jobs=1),X,y)

# 3. 同一信号只缩放数值；小数截断改变真正用于 transform 的规则。
X,y=sample_by_counts([2,4,6,8,10,12,14,16,18,19])
X['变量']=X['变量']*1e-6
for method in ['uniform','quantile','tree','best_iv']:
    for digits in [4,8]:
        fit_case('精度/'+method+'/'+str(digits),OptimalBinning(method=method,lift_refine=False,decimal=digits,max_n_bins=5,min_n_bins=2,min_bin_size=0.05,n_jobs=1,random_state=42),X,y)

# 4. 正向风险却强制反向，至少两个箱不可行；检查是否明确报不可行。
X,y=sample_by_counts([2,4,6,8,10,12,14,16,18,19])
for method in ['tree','best_iv','monotonic','cp_sat','quantile']:
    fit_case('不可行方向/'+method,OptimalBinning(method=method,lift_refine=False,max_n_bins=4,min_n_bins=3,min_bin_size=0.05,monotonic='descending',n_jobs=1,random_state=42),X,y)

# 5. min_n_bins 与 max_bin_size 在数值字段的统一验收。
X,y=sample_by_counts([20,2],[90,10])
for method in ['quantile','tree','chi','best_iv','or_tools','cp_sat']:
    fit_case('不可行大小/'+method,OptimalBinning(method=method,lift_refine=False,max_n_bins=5,min_n_bins=2,min_bin_size=0.2,n_jobs=1,random_state=42),X,y)

EVIDENCE['结构与约束反例']=CASES
save_evidence()
print(pd.DataFrame(CASES).reindex(columns=['实验','切点','声明箱数','实际箱数','箱样本数','箱坏率','错误']).to_string(index=False))


             实验                                                                                              切点   声明箱数   实际箱数                 箱样本数                                                                 箱坏率                                                                                                                                                   错误
  行顺序/cp_sat/原始                                                                                 [2.5, 5.5, 8.5] 4.0000 4.0000     [60, 60, 60, 60] [0.15, 0.31666666666666665, 0.5833333333333334, 0.8166666666666667]                                                                                                                                                  NaN
  行顺序/cp_sat/乱序                                                                                 [2.5, 5.5, 8.5] 4.0000 4.0000     [60, 60, 60, 60] [0.15, 0.31666666666666665, 0.5833333333333334, 0.8166666666666667]                                                          

## 预分箱调用链与默认选项

追踪真实算法入口调用次数，而非只比较相同输出。使用固定实例形式的等频预分箱，避免预分箱自身再触发默认 Lift 调整。

In [5]:
from sklearn.base import clone
PRE=[]
X,y=sample_by_counts([2,3,8,4,13,9,18,14,15,12,19,17])
for method,cls,entry in [('cp_sat',hbin.CPSATBinning,'_cp_sat_numerical'),('or_tools',hbin.ORBinning,'_or_numerical'),('tree',hbin.TreeBinning,'_fit_numerical'),('genetic',hbin.GeneticBinning,'_fit_numerical')]:
    original=getattr(cls,entry)
    calls=[]
    def traced(self,*args,**kwargs):
        calls.append(1)
        return original(self,*args,**kwargs)
    for use_prebin in [False,True]:
        calls.clear()
        params=dict(method=method,lift_refine=False,max_n_bins=4,min_n_bins=2,min_bin_size=0.05,n_jobs=1,random_state=42)
        if use_prebin: params['prebinning']=hbin.QuantileBinning(max_n_bins=8,min_bin_size=0.05,n_jobs=1)
        with patch.object(cls,entry,new=traced):
            b=OptimalBinning(**params).fit(X,y)
        PRE.append({'方法':method,'启用预分箱':use_prebin,'主算法调用次数':len(calls),'切点':b.splits_['变量'].tolist(),**aggregate(b.transform(X)['变量'],y)})
EVIDENCE['预分箱入口追踪']=PRE
b=OptimalBinning(method='tree',lift_refine=False,n_jobs=1)
c=clone(b)
EVIDENCE['clone选项']={'原始选项':b._fit_control_options,'clone后选项':c._fit_control_options,'get_params中存在': 'lift_refine' in b.get_params()}
print(pd.DataFrame(PRE)[['方法','启用预分箱','主算法调用次数','切点','IV']].to_string(index=False))
print('clone选项',EVIDENCE['clone选项'])
save_evidence()


      方法  启用预分箱  主算法调用次数               切点     IV
  cp_sat  False        1  [1.5, 3.5, 9.5] 1.3981
  cp_sat   True        1 [4.0, 5.5, 10.0] 1.3688
or_tools  False        1  [1.5, 3.5, 9.5] 1.3981
or_tools   True        1 [4.0, 5.5, 10.0] 1.3688
    tree  False        1  [3.5, 5.5, 9.5] 1.3688
    tree   True        1 [4.0, 5.5, 10.0] 1.3688
 genetic  False        1 [4.0, 9.0, 10.0] 1.3035
 genetic   True        1 [4.0, 5.5, 10.0] 1.3688
clone选项 {'原始选项': {'lift_refine': False}, 'clone后选项': {'lift_refine': False}, 'get_params中存在': True}


## 搜索边界、参数保留、卡方和预分箱最优性

保留独立原始行核算与小候选集穷举；追踪真实 CpSolver 调用。

In [6]:
from itertools import combinations
from sklearn.base import clone
DETAIL={}
# A. decimal 在部分直接分箱器的 sklearn clone 中丢失。
precision=[]
x=pd.DataFrame({'变量':np.arange(100,dtype=float)/100})
y=pd.Series(np.tile([0,1],50))
for method,cls in OptimalBinning._method_classes().items():
    obj=cls(decimal=8,n_jobs=1)
    precision.append({'方法':method,'原始decimal':obj.decimal,'clone后decimal':clone(obj).decimal})
DETAIL['精度clone']=precision
print('精度丢失', [r for r in precision if r['原始decimal']!=r['clone后decimal']])

# B. 搜索的 <= 与 transform 的 < 在重复值上不是同一分区。
x=np.repeat([0.,1.,2.],10)
y=np.concatenate([np.r_[np.ones(k,dtype=int),np.zeros(10-k,dtype=int)] for k in [1,7,8]])
b=hbin.BestIVBinning(min_bin_size=1,n_jobs=1)
cb=np.cumsum(y); cg=np.arange(1,len(y)+1)-cb
search_iv=b._calc_iv_fast(x,y,cg,cb,int((y==0).sum()),int(y.sum()),[1.])
actual=aggregate(np.searchsorted([1.],x,side='right'),y)
DETAIL['搜索与应用边界']={'切点':1.,'搜索IV':search_iv,'最终分箱统计':actual,'搜索左箱人数':int((x<=1).sum()),'应用左箱人数':int((x<1).sum())}
print('搜索边界',DETAIL['搜索与应用边界'])

# C. 卡方阈值提前停止后，_adjust_bins 从左边直接删除切点。
X=pd.DataFrame({'变量':np.repeat(np.arange(8,dtype=float),100)})
y=pd.Series(np.concatenate([np.r_[np.ones(k,dtype=int),np.zeros(100-k,dtype=int)] for k in [10,90,20,80,30,70,40,60]]))
chi=[]
for threshold in [0.000001,3.841,1e9]:
    stages={}
    original=hbin.ChiMergeBinning._adjust_bins
    def traced(self,x,y,splits):
        stages['调整前切点']=splits.tolist()
        out=original(self,x,y,splits)
        stages['调整后切点']=out.tolist()
        return out
    with patch.object(hbin.ChiMergeBinning,'_adjust_bins',new=traced):
        b=hbin.ChiMergeBinning(max_n_bins=4,min_n_bins=2,min_bin_size=0.05,min_chi2_threshold=threshold,n_jobs=1).fit(X,y)
    chi.append({'阈值':threshold,**stages,'最终切点':b.splits_['变量'].tolist(),**aggregate(b.transform(X)['变量'],y)})
DETAIL['卡方裁剪']=chi
print('卡方裁剪',chi)

# D. 主算法结果映射到预分箱后未做候选最优解搜索。
X=pd.DataFrame({'变量':np.repeat(np.arange(12,dtype=float),20)})
y=pd.Series(np.concatenate([np.r_[np.ones(k,dtype=int),np.zeros(20-k,dtype=int)] for k in [2,3,8,4,13,9,18,14,15,12,19,17]]))
pre=hbin.QuantileBinning(max_n_bins=8,min_bin_size=0.05,n_jobs=1).fit(X,y)
candidates=pre.splits_['变量']
b=OptimalBinning(method='cp_sat',prebinning=pre,lift_refine=False,max_n_bins=4,min_n_bins=2,min_bin_size=0.05,monotonic=False,n_jobs=1).fit(X,y)
best=None
for n_cuts in range(1,4):
    for cuts in combinations(candidates,n_cuts):
        bins=np.searchsorted(cuts,X['变量'],side='right')
        raw=pd.DataFrame({'箱':bins,'y':y}).groupby('箱')['y'].agg(['size','sum'])
        if raw['size'].min()<12: continue
        good=raw['size']-raw['sum']; bad=raw['sum']
        pbad=bad/bad.sum(); pgood=good/good.sum()
        iv=float(((pbad-pgood)*np.log(pbad/pgood)).sum())
        if best is None or iv>best['IV']: best={'切点':list(cuts),'IV':iv}
opt=OptBinning(user_splits=candidates,max_n_bins=4,min_n_bins=2,min_bin_size=0.05,monotonic_trend=None).fit(X['变量'].to_numpy(),y.to_numpy())
DETAIL['预分箱候选最优性']={'候选':candidates.tolist(),'hscredit切点':b.splits_['变量'].tolist(),'hscreditIV':aggregate(b.transform(X)['变量'],y)['IV'],'穷举最优':best,'optbinning切点':opt.splits.tolist(),'optbinning状态':opt.status,'optbinningIV':aggregate(opt.transform(X['变量'],metric='indices'),y)['IV']}
print('候选最优性',DETAIL['预分箱候选最优性'])

# E. 核验修复后的数值主路径创建了真正的 CpSolver。
from ortools.sat.python.cp_model import CpSolver
with patch('ortools.sat.python.cp_model.CpSolver',wraps=CpSolver) as solver:
    obj=OptimalBinning(method='cp_sat',lift_refine=False,max_n_bins=4,min_n_bins=2,min_bin_size=0.05,monotonic=False,n_jobs=1).fit(X,y)
    DETAIL['CP求解器调用次数']=solver.call_count
print('CP求解器调用次数',DETAIL['CP求解器调用次数'])

# F. 只赋值、未在本类或共享调用链使用的参数：静态 AST 读写统计。
unused=[]
for file,attrs in {'target_bad_rate_binning.py':['strict_mode'],'smooth_binning.py':['merge_criterion'],'kernel_density_binning.py':['iv_weight'],'best_lift_binning.py':['min_lift'],'or_binning.py':['n_prebins'],'cp_sat_binning.py':['num_workers']}.items():
    tree=ast.parse((ROOT/'hscredit/core/binning'/file).read_text(encoding='utf-8'))
    for attr in attrs:
        reads=[node.lineno for node in ast.walk(tree) if isinstance(node,ast.Attribute) and node.attr==attr and isinstance(node.ctx,ast.Load)]
        writes=[node.lineno for node in ast.walk(tree) if isinstance(node,ast.Attribute) and node.attr==attr and isinstance(node.ctx,ast.Store)]
        unused.append({'文件':file,'参数':attr,'读取行':reads,'赋值行':writes})
DETAIL['参数使用静态扫描']=unused
print('未读取参数',unused)
EVIDENCE['专项证据']=DETAIL
save_evidence()


精度丢失 []
搜索边界 {'切点': 1.0, '搜索IV': 1.9127624668775125, '最终分箱统计': {'实际箱数': 2, '最小箱占比': 0.3333333333333333, '最大箱占比': 0.6666666666666666, 'IV': 1.9127624668775125, 'KS': 0.5803571428571428, '箱样本数': [10, 20], '箱坏样本数': [1, 15], '箱坏率': [0.1, 0.75]}, '搜索左箱人数': 20, '应用左箱人数': 10}
卡方裁剪 [{'阈值': 1e-06, '调整前切点': [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5], '调整后切点': [0.5, 1.5, 2.5], '最终切点': [0.5, 1.5, 2.5], '实际箱数': 4, '最小箱占比': 0.125, '最大箱占比': 0.625, 'IV': 1.1230082936250048, 'KS': 0.19999999999999996, '箱样本数': [100, 100, 100, 500], '箱坏样本数': [10, 90, 20, 280], '箱坏率': [0.1, 0.9, 0.2, 0.56]}, {'阈值': 3.841, '调整前切点': [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5], '调整后切点': [0.5, 1.5, 2.5], '最终切点': [0.5, 1.5, 2.5], '实际箱数': 4, '最小箱占比': 0.125, '最大箱占比': 0.625, 'IV': 1.1230082936250048, 'KS': 0.19999999999999996, '箱样本数': [100, 100, 100, 500], '箱坏样本数': [10, 90, 20, 280], '箱坏率': [0.1, 0.9, 0.2, 0.56]}, {'阈值': 1000000000.0, '调整前切点': [0.5, 1.5, 2.5], '调整后切点': [0.5, 1.5, 2.5], '最终切点': [0.5, 1.5, 2.5], '实际箱数': 4, '最小箱占比': 0.125, '最大箱占比'

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be rem

候选最优性 {'候选': [1.0, 2.75, 4.0, 5.5, 7.0, 8.25, 10.0], 'hscredit切点': [4.0, 5.5, 10.0], 'hscreditIV': 1.3687899576175322, '穷举最优': {'切点': [np.float64(4.0), np.float64(5.5), np.float64(10.0)], 'IV': 1.3687899576175322}, 'optbinning切点': [4.0, 5.5, 10.0], 'optbinning状态': 'OPTIMAL', 'optbinningIV': 1.3687899576175322}
CP求解器调用次数 1
未读取参数 [{'文件': 'target_bad_rate_binning.py', '参数': 'strict_mode', '读取行': [309], '赋值行': [194]}, {'文件': 'smooth_binning.py', '参数': 'merge_criterion', '读取行': [518, 528, 530], '赋值行': [216]}, {'文件': 'kernel_density_binning.py', '参数': 'iv_weight', '读取行': [612, 612, 628, 628], '赋值行': [225]}, {'文件': 'best_lift_binning.py', '参数': 'min_lift', '读取行': [], '赋值行': [210]}, {'文件': 'or_binning.py', '参数': 'n_prebins', '读取行': [448], '赋值行': [348]}, {'文件': 'cp_sat_binning.py', '参数': 'num_workers', '读取行': [414], '赋值行': [260]}]


## 留出集比较

5 个固定种子，各按 FPD 分层划分训练集 70% / 留出集 30%。分箱与每箱风险均只在训练集拟合，测试 AUC 使用训练箱坏率，未利用测试标签翻转方向。重复切分用于敏感性观察，不是五份独立样本，也不替代按客户和时间设计的 OOT 验证。

In [7]:
HOLDOUT=[]
configs=[('hs/tree/启用调整','hs','tree',True),('hs/tree/关闭','hs','tree',False),('hs/best_iv/启用调整','hs','best_iv',True),('hs/best_iv/关闭','hs','best_iv',False),('hs/chi/关闭','hs','chi',False),('hs/cp_sat/关闭','hs','cp_sat',False),('toad/dt','toad','dt',False),('toad/chi','toad','chi',False),('optbinning/无单调','opt',None,False),('optbinning/auto','opt','auto',False)]
for feature in FEATURES:
    for seed in [7,19,42,83,137]:
        train,test=train_test_split(np.arange(len(DATA)),test_size=.3,stratify=DATA.FPD,random_state=seed)
        Xtr=DATA.iloc[train][[feature]].reset_index(drop=True); Xte=DATA.iloc[test][[feature]].reset_index(drop=True)
        ytr=DATA.iloc[train].FPD.reset_index(drop=True); yte=DATA.iloc[test].FPD.reset_index(drop=True)
        for label,family,method,refine in configs:
            row={'字段':feature,'种子':seed,'方法':label,'训练行数':len(train),'留出行数':len(test)}
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                try:
                    if family=='hs':
                        b=OptimalBinning(method=method,lift_refine=refine,**COMMON).fit(Xtr,ytr)
                        tr=b.transform(Xtr)[feature].to_numpy(); te=b.transform(Xte)[feature].to_numpy()
                    elif family=='toad':
                        kw={'n_bins':5}
                        if method=='dt': kw.update(min_samples=.05,random_state=42)
                        tr,splits=toad.merge(Xtr[feature].to_numpy(),ytr.to_numpy(),method=method,return_splits=True,**kw)
                        te=np.searchsorted(splits,Xte[feature],side='right')
                    else:
                        b=OptBinning(max_n_bins=5,min_n_bins=2,min_bin_size=.05,max_n_prebins=20,min_prebin_size=.05,monotonic_trend=method,time_limit=30).fit(Xtr[feature].to_numpy(),ytr.to_numpy())
                        tr=b.transform(Xtr[feature],metric='indices'); te=b.transform(Xte[feature],metric='indices')
                    train_stats=aggregate(tr,ytr); test_stats=aggregate(te,yte)
                    risk=pd.DataFrame({'箱':tr,'标签':ytr}).groupby('箱')['标签'].mean()
                    train_pred=pd.Series(tr).map(risk).fillna(ytr.mean())
                    test_pred=pd.Series(te).map(risk).fillna(ytr.mean())
                    row.update({'训练IV':train_stats['IV'],'留出IV':test_stats['IV'],'训练AUC':roc_auc_score(ytr,train_pred),'留出AUC':roc_auc_score(yte,test_pred),'训练实际箱数':train_stats['实际箱数'],'训练最小箱占比':train_stats['最小箱占比']})
                except Exception as error:
                    row['错误']=type(error).__name__+': '+str(error)
                row['警告']=list(dict.fromkeys(str(w.message) for w in caught))
            HOLDOUT.append(row)
    print('完成留出集',feature)
EVIDENCE['留出集比较']=HOLDOUT
save_evidence()
print(pd.DataFrame(HOLDOUT).groupby(['字段','方法'])[['训练IV','训练AUC','留出AUC']].agg(['mean','std']).rename(columns={'mean':'均值','std':'标准差'},level=1).to_string())


完成留出集 衡枢鉴真分老客版


完成留出集 近六个月非银多头机构数


完成留出集 青云24
                              训练IV         训练AUC         留出AUC       
                                均值    标准差     均值    标准差     均值    标准差
字段          方法                                                       
衡枢鉴真分老客版    hs/best_iv/关闭   0.3788 0.0498 0.6496 0.0160 0.5895 0.0404
            hs/best_iv/启用调整 0.2872 0.0420 0.6250 0.0078 0.6186 0.0177
            hs/chi/关闭       0.2076 0.1416 0.6024 0.0527 0.5631 0.0335
            hs/cp_sat/关闭    0.3601 0.0658 0.6492 0.0181 0.5876 0.0396
            hs/tree/关闭      0.3946 0.0467 0.6535 0.0118 0.5834 0.0412
            hs/tree/启用调整    0.2976 0.0441 0.6278 0.0083 0.6257 0.0088
            optbinning/auto 0.3232 0.0415 0.6304 0.0100 0.6099 0.0283
            optbinning/无单调  0.4181 0.0549 0.6571 0.0120 0.5703 0.0307
            toad/chi        0.4898 0.1761 0.6391 0.0127 0.6337 0.0238
            toad/dt         0.3946 0.0467 0.6535 0.0118 0.5834 0.0412
近六个月非银多头机构数 hs/best_iv/关闭   0.1395 0.0373 0.5915 0.0132 0.5268 0.0253
         

## 类别字段、二维分箱和 MDLP 补充

类别字段使用约定的 `商品类别`。二维部分验证真实数据人口守恒并审阅合并流程，不把连通区域贪心合并等同于 optbinning 的二维数学规划。MDLP 补充展示特殊值与强制补箱的行为。

In [8]:
EXTRA={}
catrows=[]
for method in OptimalBinning.VALID_METHODS:
    row={'方法':method}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        try:
            b=OptimalBinning(method=method,**COMMON).fit(DATA[['商品类别']],DATA.FPD)
            bins=b.transform(DATA[['商品类别']])['商品类别']
            row.update(aggregate(bins,DATA.FPD));row['类别分组']=b.splits_['商品类别']
            row['未知类别箱号']=int(b.transform(pd.DataFrame({'商品类别':['审计专用未知类别']})).iloc[0,0])
        except Exception as error: row['错误']=type(error).__name__+': '+str(error)
        row['警告']=list(dict.fromkeys(str(w.message) for w in caught))
    catrows.append(row)
EXTRA['类别字段矩阵']=catrows
print(pd.DataFrame(catrows).reindex(columns=['方法','实际箱数','IV','错误']).to_string(index=False))

b2=hbin.OptimalBinning2D(max_n_bins=5,max_n_bins_2d=5,min_bin_size=.05,n_jobs=1,random_state=42).fit(DATA[FEATURES[:2]],DATA.FPD)
bins2=np.asarray(b2.transform(DATA[FEATURES[:2]],metric='indices')).reshape(-1)
EXTRA['二维真实样本']={'轴1切点':b2.splits_x_,'轴2切点':b2.splits_y_,'合并网格':b2.solution_,**aggregate(bins2,DATA.FPD)}
print('二维真实样本',EXTRA['二维真实样本'])

# MDLP 的 force_min_bins 为真时，补箱循环实际上以 max_n_bins-1 为目标。
x=np.arange(200,dtype=float)
y=np.tile([0,1],100)
mdlp=[]
for force in [False,True]:
    b=hbin.MDLPBinning(max_n_bins=5,min_n_bins=2,min_bin_size=1,force_min_bins=force,n_jobs=1)
    raw=b._mdlp_split_v3(x,y)
    fitted=b.fit(pd.DataFrame({'变量':x}),pd.Series(y))
    mdlp.append({'强制最小箱':force,'算法原始切点':raw,'后处理切点':fitted.splits_['变量'].tolist()})
EXTRA['MDLP强制补箱']=mdlp
print('MDLP补箱',mdlp)

# 特殊值不应参与普通区间切点的目标搜索（固定绝对最小样本数）。
base_x=np.repeat(np.arange(12,dtype=float),20)
base_y=np.concatenate([np.r_[np.ones(k,dtype=int),np.zeros(20-k,dtype=int)] for k in [2,3,8,4,13,9,18,14,15,12,19,17]])
special=[]
for extra_count in [0,200]:
    xs=np.r_[base_x,np.repeat(-999.,extra_count)]
    ys=np.r_[base_y,np.ones(extra_count,dtype=int)]
    b=hbin.MDLPBinning(max_n_bins=3,min_n_bins=2,min_bin_size=1,special_codes=[-999],force_min_bins=False,n_jobs=1).fit(pd.DataFrame({'变量':xs}),pd.Series(ys))
    special.append({'新增特殊值行数':extra_count,'切点':b.splits_['变量'].tolist(),'普通样本分箱':aggregate(b.transform(pd.DataFrame({'变量':base_x}))['变量'],base_y)})
EXTRA['MDLP特殊值']=special
print('MDLP特殊值',special)

# 方法参数端到端检查；不适用于全样本分箱的 min_lift 非零值应被明确拒绝。
param_cases=[]
for method,param,values in [('target_bad_rate','strict_mode',[True,False]),('best_lift','min_lift',[0.,4.]),('smooth','merge_criterion',['iv','chi2']),('kernel_density','iv_weight',[0.,1.]),('or_tools','n_prebins',[5,50])]:
    splits=[]
    for value in values:
        try:
            b=OptimalBinning(method=method,lift_refine=False,**COMMON,**{param:value}).fit(DATA[[FEATURES[0]]],DATA.FPD)
            splits.append(b.splits_[FEATURES[0]].tolist())
        except Exception as error:
            splits.append({'异常类型':type(error).__name__,'异常':str(error)})
    param_cases.append({'方法':method,'参数':param,'取值':values,'两次切点':splits,'规则完全一致':splits[0]==splits[1]})
EXTRA['参数敏感性']=param_cases
print('参数敏感性',param_cases)
EVIDENCE['补充实验']=EXTRA
save_evidence()


             方法  实际箱数     IV  错误
        uniform     3 0.0357 NaN
       quantile     2 0.0235 NaN
           tree     3 0.0357 NaN
            chi     3 0.0357 NaN
        best_ks     3 0.0357 NaN
        best_iv     3 0.0357 NaN
           mdlp     2 0.0323 NaN
       or_tools     3 0.0357 NaN
         cp_sat     3 0.0357 NaN
           cart     3 0.0357 NaN
         kmeans     3 0.0357 NaN
      monotonic     3 0.0357 NaN
        genetic     3 0.0357 NaN
         smooth     3 0.0357 NaN
 kernel_density     3 0.0357 NaN
      best_lift     2 0.0323 NaN
target_bad_rate     3 0.0357 NaN
二维真实样本 {'轴1切点': array([0.04776716, 0.07192072, 0.09840784, 0.13415437]), '轴2切点': array([50., 58., 64., 71.]), '合并网格': array([[0, 0, 0, 1, 1],
       [0, 0, 0, 1, 2],
       [0, 0, 0, 0, 2],
       [0, 0, 3, 0, 0],
       [3, 3, 3, 3, 4]]), '实际箱数': 5, '最小箱占比': 0.058762886597938144, '最大箱占比': 0.5865979381443299, 'IV': 0.2511652866016394, 'KS': 0.22213993511073488, '箱样本数': [569, 87, 76, 181, 57], '箱坏样本数': [

参数敏感性 [{'方法': 'target_bad_rate', '参数': 'strict_mode', '取值': [True, False], '两次切点': [[0.12592262562939105, 0.13779788928375, 0.15893490950186315, 0.19172932521007985], [0.12592262562939105, 0.13779788928375, 0.15893490950186315, 0.19172932521007985]], '规则完全一致': True}, {'方法': 'best_lift', '参数': 'min_lift', '取值': [0.0, 4.0], '两次切点': [[0.06912995898451912, 0.0751351105180246, 0.14682411085362995, 0.1914799087349981], {'异常类型': 'ValueError', '异常': 'min_lift 非零的箱过滤不适用于覆盖全部样本的分箱器；请在分箱表或规则分析中按 LIFT 筛选'}], '规则完全一致': False}, {'方法': 'smooth', '参数': 'merge_criterion', '取值': ['iv', 'chi2'], '两次切点': [[0.05764420532478552, 0.10554738252388007, 0.11573927837499934, 0.1534061268117138], [0.05089360219122456, 0.10554738252388007, 0.11573927837499934, 0.1534061268117138]], '规则完全一致': False}, {'方法': 'kernel_density', '参数': 'iv_weight', '取值': [0.0, 1.0], '两次切点': [[0.035678229467299524, 0.06971702428187693, 0.07592794293194188, 0.1569840054163171], [0.035678229467299524, 0.06971702428187693, 0.075927942931941

## 显式目标坏率与验证元信息

保存显式目标坏率参数失败的完整异常类型、命令执行范围和依赖源码指纹。已有测试通过只能证明这些用例没有发现问题，不能覆盖上述反例。

In [9]:
from xml.etree import ElementTree as ET
import subprocess
contract=[]
for entry in ['直接类','统一入口']:
    params={'target_bad_rates':[.1,.5,.9],'strict_mode':True,'max_n_bins':3,'min_n_bins':2,'min_bin_size':.1,'n_jobs':1}
    b=hbin.TargetBadRateBinning(**params) if entry=='直接类' else OptimalBinning(method='target_bad_rate',**params)
    try:
        b.fit(DATA[[FEATURES[0]]],DATA.FPD)
        contract.append({'入口':entry,'完成':True})
    except Exception as error:
        contract.append({'入口':entry,'完成':False,'异常类型':type(error).__name__,'异常':str(error)})
EVIDENCE['显式目标坏率参数']=contract
print('显式目标坏率',contract)

EVIDENCE['HEAD']=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
EVIDENCE['分箱模块未提交差异']=subprocess.check_output(['git','status','--short','--','hscredit/core/binning'],cwd=ROOT,text=True).strip()
source_paths=[Path(toad.__file__).parent/'merge.pyx',Path(inspect.getfile(OptBinning)).parent/'cp.py',ROOT/'hscredit/core/metrics/_binning.py',ROOT/'hscredit/core/models/scorecard/scorecard.py']
EVIDENCE['统计与对照实现SHA256']={str(path):hashlib.sha256(path.read_bytes()).hexdigest() for path in source_paths}
for relative,digest in EVIDENCE['源码SHA256'].items():
    assert hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()==digest, '审计期间分箱源码发生变更'
EVIDENCE['测试记录']=[]
for file,scope in [('binning-repairs-all-second.xml','tests/test_binning/'),('binning-repairs-full.xml','tests/（中间版本）'),('binning-repairs-final.xml','最终分箱和评分卡相关模块')]:
    xml_path=ROOT/'.audit_tmp'/file
    if xml_path.exists():
        suite=ET.parse(xml_path).getroot().find('testsuite')
        record={'日期':'2026-10-27','解释器':sys.executable,'范围':scope,
            **{key:int(suite.get(key,0)) for key in ['tests','failures','errors','skipped']},'耗时秒':float(suite.get('time'))}
        EVIDENCE['测试记录'].append(record)
        print('测试记录',record)
full=ET.parse(ROOT/'.audit_tmp/binning-repairs-full.xml').getroot()
final=ET.parse(ROOT/'.audit_tmp/binning-repairs-final.xml').getroot()
failed={(t.get('classname'),t.get('name')) for t in full.iter('testcase') if t.find('failure') is not None or t.find('error') is not None}
passed={(t.get('classname'),t.get('name')) for t in final.iter('testcase') if t.find('failure') is None and t.find('error') is None and t.find('skipped') is None}
assert failed <= passed, failed-passed
assert int(final.find('testsuite').get('failures'))==0 and int(final.find('testsuite').get('errors'))==0
EVIDENCE['全仓失败项已逐项复测通过']=[{'测试类':a,'测试项':b} for a,b in sorted(failed)]
save_evidence()
print('证据文件',OUT/'binning-method-repair-2026-10-27.json')
print('分箱源文件数',len(EVIDENCE['源码SHA256']))
print('HEAD',EVIDENCE['HEAD'])


显式目标坏率 [{'入口': '直接类', '完成': True}, {'入口': '统一入口', '完成': True}]
测试记录 {'日期': '2026-10-27', '解释器': 'E:\\workspace\\hscredit\\.audit_tmp\\defect-repairs\\runtime\\Scripts\\python.exe', '范围': 'tests/test_binning/', 'tests': 1026, 'failures': 0, 'errors': 0, 'skipped': 0, '耗时秒': 107.309}
测试记录 {'日期': '2026-10-27', '解释器': 'E:\\workspace\\hscredit\\.audit_tmp\\defect-repairs\\runtime\\Scripts\\python.exe', '范围': 'tests/（中间版本）', 'tests': 6829, 'failures': 3, 'errors': 0, 'skipped': 9, '耗时秒': 1206.784}
测试记录 {'日期': '2026-10-27', '解释器': 'E:\\workspace\\hscredit\\.audit_tmp\\defect-repairs\\runtime\\Scripts\\python.exe', '范围': '最终分箱和评分卡相关模块', 'tests': 1190, 'failures': 0, 'errors': 0, 'skipped': 0, '耗时秒': 189.507}
证据文件 E:\workspace\hscredit\docs\validation\binning-method-repair-2026-10-27.json
分箱源文件数 26
HEAD 17e212c3641c6bafd21f79b48e9574ba1f97b76b


## 对齐树分箱的叶节点人数与切点精度

把 5% 转成相同的绝对人数 `ceil(n*0.05)`，关闭统一入口的 Lift 调整，使用 12 位切点精度，检验树算法自身能否与 toad 一致。toad ChiMerge 的样本量停止条件另行记录，不宣称与 hscredit 的硬约束等价。

In [10]:
ALIGNED=[]
for feature in FEATURES:
    count=int(np.ceil(len(DATA)*.05))
    b=OptimalBinning(method='tree',max_n_bins=5,min_n_bins=2,min_bin_size=count,min_samples_leaf=count,lift_refine=False,decimal=12,monotonic=False,random_state=42,n_jobs=1).fit(DATA[[feature]],DATA.FPD)
    own=b.transform(DATA[[feature]])[feature].to_numpy()
    other,splits=toad.merge(DATA[feature].to_numpy(),DATA.FPD.to_numpy(),method='dt',n_bins=5,min_samples=count,random_state=42,return_splits=True)
    ALIGNED.append({'字段':feature,'叶节点最小人数':count,'两库箱号完全一致':bool(np.array_equal(own,other)),'hscredit切点':b.splits_[feature].tolist(),'toad切点':splits.tolist(),'hscreditIV':aggregate(own,DATA.FPD)['IV'],'toadIV':aggregate(other,DATA.FPD)['IV']})
EVIDENCE['树分箱完全对齐']=ALIGNED
CHIALIGN=[]
for feature in FEATURES:
    for balance in [True,False]:
        bins,splits=toad.merge(DATA[feature].to_numpy(),DATA.FPD.to_numpy(),method='chi',n_bins=5,min_samples=.05,balance=balance,return_splits=True)
        CHIALIGN.append({'字段':feature,'balance':balance,'n_bins参数':5,'min_samples参数':.05,'切点':splits.tolist(),**aggregate(bins,DATA.FPD)})
EVIDENCE['toad卡方补充']=CHIALIGN
save_evidence()
print('树对齐',ALIGNED)
print('卡方最小箱参数',pd.DataFrame(CHIALIGN)[['字段','balance','实际箱数','最小箱占比','IV']].to_string(index=False))


树对齐 [{'字段': '衡枢鉴真分老客版', '叶节点最小人数': 49, '两库箱号完全一致': True, 'hscredit切点': [0.06950388103723526, 0.07573091983795166, 0.15583351254463196, 0.19294103235006332], 'toad切点': [0.06950388103723526, 0.07573091983795166, 0.15583351254463196, 0.19294103235006332], 'hscreditIV': 0.33675124448737387, 'toadIV': 0.33675124448737387}, {'字段': '近六个月非银多头机构数', '叶节点最小人数': 49, '两库箱号完全一致': True, 'hscredit切点': [45.5, 64.5, 71.5, 74.5], 'toad切点': [45.5, 64.5, 71.5, 74.5], 'hscreditIV': 0.10086291871873754, 'toadIV': 0.10086291871873754}, {'字段': '青云24', '叶节点最小人数': 49, '两库箱号完全一致': True, 'hscredit切点': [553.5, 573.5, 608.5, 622.5], 'toad切点': [553.5, 573.5, 608.5, 622.5], 'hscreditIV': 0.2585628007361136, 'toadIV': 0.2585628007361136}]
卡方最小箱参数          字段  balance  实际箱数  最小箱占比     IV
   衡枢鉴真分老客版     True     5 0.0433 0.3262
   衡枢鉴真分老客版    False     5 0.0021 1.0528
近六个月非银多头机构数     True     5 0.0062 0.1120
近六个月非银多头机构数    False     5 0.0010 0.2697
       青云24     True     5 0.0505 1.5806
       青云24    False     5 0.00

## 修复验收断言

验证核心反例已消除，并与修复前默认配置比较。训练 IV 变化仅代表当前样本上的规则差异，泛化表现仍查阅前面的留出结果。

In [11]:
assert all('错误' not in row for row in EVIDENCE['真实数据完整样本比较'])
assert all('错误' not in row for row in EVIDENCE['留出集比较'])
for method in ['cp_sat','or_tools']:
    rows=[row for row in EVIDENCE['结构与约束反例'] if row['实验'].startswith('行顺序/'+method+'/')]
    assert len(rows)==3 and all(row['实际箱数']==4 for row in rows)
    assert all(row['切点']==rows[0]['切点'] for row in rows)
for row in EVIDENCE['结构与约束反例']:
    if row['实验'].startswith('不可行'):
        assert '错误' in row, row
    elif row['实验'].startswith('低基数'):
        assert row['实际箱数']==3, row
    elif row['实验'].startswith('精度'):
        assert row['实际箱数']==row['声明箱数'] and row['实际箱数']>=2, row
assert all(row['原始decimal']==row['clone后decimal']==8 for row in EVIDENCE['专项证据']['精度clone'])
assert EVIDENCE['clone选项']['原始选项']==EVIDENCE['clone选项']['clone后选项']
assert EVIDENCE['专项证据']['CP求解器调用次数']>=1
opt=EVIDENCE['专项证据']['预分箱候选最优性']
np.testing.assert_allclose(opt['hscreditIV'],opt['穷举最优']['IV'],atol=1e-8)
assert all(row['两库箱号完全一致'] for row in EVIDENCE['树分箱完全对齐'])
original=json.loads((OUT/'binning-method-audit-2026-10-27.json').read_text(encoding='utf-8'))
comparison=[]
for feature in FEATURES:
    for method in ['tree','chi','best_iv','cp_sat','or_tools']:
        before=next(row for row in original['真实数据完整样本比较'] if row['字段']==feature and row['方法']==method+'/默认调整' and row['库']=='hscredit')
        after=next(row for row in EVIDENCE['真实数据完整样本比较'] if row['字段']==feature and row['方法']==method+'/关闭调整' and row['库']=='hscredit')
        comparison.append({'字段':feature,'方法':method,'修复前默认IV':before['IV'],'修复后默认IV':after['IV'],
            '修复后实际箱数':after['实际箱数'],'修复后最小箱占比':after['最小箱占比']})
EVIDENCE['默认配置前后对比']=comparison
EVIDENCE['核心反例断言全部通过']=True
save_evidence()
print(pd.DataFrame(comparison).to_string(index=False))
print('核心反例、原始计数核验和前后对比全部完成。')


         字段       方法  修复前默认IV  修复后默认IV  修复后实际箱数  修复后最小箱占比
   衡枢鉴真分老客版     tree   0.2770   0.3368        5    0.0505
   衡枢鉴真分老客版      chi   0.2126   0.1249        4    0.0907
   衡枢鉴真分老客版  best_iv   0.2870   0.3177        5    0.0598
   衡枢鉴真分老客版   cp_sat   0.1135   0.2914        5    0.0598
   衡枢鉴真分老客版 or_tools   0.2468   0.2840        5    0.0505
近六个月非银多头机构数     tree   0.0754   0.1009        5    0.0619
近六个月非银多头机构数      chi   0.0721   0.0680        2    0.1309
近六个月非银多头机构数  best_iv   0.0692   0.1178        5    0.0577
近六个月非银多头机构数   cp_sat   0.0044   0.1178        5    0.0577
近六个月非银多头机构数 or_tools   0.0859   0.0915        5    0.0660
       青云24     tree   0.0866   0.2586        5    0.0794
       青云24      chi   0.0475   0.0167        3    0.2825
       青云24  best_iv   0.0808   0.2147        5    0.0784
       青云24   cp_sat   0.1046   0.1808        5    0.0629
       青云24 or_tools   0.2034   0.1413        5    0.0536
核心反例、原始计数核验和前后对比全部完成。
